# GHZ Benchmark Tutorial


This notebook demonstrates the `GHZBenchmark` class. It walks through how to:
1. configure the benchmark
2.  generate GHZ witness circuits
3.  attach synthetic results
4.  evaluate fidelities
5.  inspect both witness and shadow measurement options

In [1]:
from frontier import GHZBenchmark, evaluate_fidelity, certify_fidelity_gt_half

## Create a GHZ benchmark instance
Configure the number of qubits, fanout topology, and measurement scheme. Here we focus on the witness protocol with a star topology.


In [2]:
n_qubits = 5
sample_size = 3
shots = 2048

benchmark = GHZBenchmark(
    number_of_qubits=n_qubits,
    sample_size=sample_size,
    preparation_method="star",
    measurement_scheme="witness",
    format="qasm2",
    target_sdk="qiskit",
    shots=shots,
    benchmark_metadata={"type": "GHZ fidelity"},
    auto_save=False,
    workdir="tutorial_data",
)

## Generate benchmark circuits
`create_benchmark()` builds the GHZ preparation and witness circuits for every sample.


In [3]:
benchmark.create_benchmark()
benchmark  # display summary

GHZBenchmark(
  benchmark_id='ghz_nq5_s3_20260409T141737_e9156893',
  number_of_qubits=5,
  sample_size=3,
  format='qasm2',
  target_sdk='qiskit',
  workdir=WindowsPath('tutorial_data'),
)

## Inspect configuration metadata


In [4]:
print("benchmark.benchmark_id:", benchmark.benchmark_id)
print("measurement_scheme:", benchmark.measurement_scheme)
print("preparation_method:", benchmark.preparation_method)
print("number_of_measurements:", benchmark.number_of_measurements)
print("emitter_options:", benchmark.emitter_options)

benchmark.benchmark_id: ghz_nq5_s3_20260409T141737_e9156893
measurement_scheme: witness
preparation_method: star
number_of_measurements: 2
emitter_options: QasmEmitterOptions(format='qasm2', target_sdk='qiskit', includes=('qelib1.inc',), float_precision=6, custom_template=None, map={'x': 'x', 'y': 'y', 'z': 'z', 'h': 'h', 's': 's', 'sdg': 'sdg', 't': 't', 'u': 'u3', 'cx': 'cx', 'cy': 'cy', 'cz': 'cz', 'swap': 'swap', 'rx': 'rx', 'ry': 'ry', 'rz': 'rz', 'measure': 'measure'})


## Sample structure
Each sample contains two witness circuits (X-basis and Z-basis) plus per-circuit metadata.


In [5]:
sample = benchmark.samples[0]
print("sample_id:", sample["sample_id"])
print("sample metadata:", sample["sample_metadata"])
for circuit in sample["circuits"]:
    print("- circuit_id:", circuit["circuit_id"])
    print("  basis:", circuit["metadata"].get("basis"))
    print("  qasm preview:")
    print(circuit["qasm"][:200] + "...")

sample_id: 0
sample metadata: {'type': 'ghz', 'preparation_method': 'star', 'measurement_scheme': 'witness', 'number_of_measurements': 2}
- circuit_id: 0_witness_x
  basis: x
  qasm preview:
OPENQASM 2.0;

include "qelib1.inc";

qreg q[5];
creg c[5];

h q[0];
cx q[0], q[1];
cx q[0], q[2];
cx q[0], q[3];
cx q[0], q[4];
h q[0];
h q[1];
h q[2];
h q[3];
h q[4];

measure q[0] -> c[0];
measure ...
- circuit_id: 0_witness_z
  basis: z
  qasm preview:
OPENQASM 2.0;

include "qelib1.inc";

qreg q[5];
creg c[5];

h q[0];
cx q[0], q[1];
cx q[0], q[2];
cx q[0], q[3];
cx q[0], q[4];

measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[2] -> c[2];
meas...


## Retrieve circuit payloads
Use helper methods to list every circuit ID or grab the serialized circuits.


## Witness circuit QASM preview
The first sample always contains two witness circuits (X-basis parity and Z-basis populations).
Inspect their circuit IDs and headers directly from the generated benchmark object.


In [6]:
witness_map = {}
for circuit in benchmark.samples[0]["circuits"]:
    basis = circuit["metadata"].get("basis")
    witness_map[basis] = circuit
    print(f"Basis {basis.upper()} -> {circuit['circuit_id']}")
    print("\n".join(circuit["qasm"].splitlines()[:8]), "...\n")

Basis X -> 0_witness_x
OPENQASM 2.0;

include "qelib1.inc";

qreg q[5];
creg c[5];

h q[0]; ...

Basis Z -> 0_witness_z
OPENQASM 2.0;

include "qelib1.inc";

qreg q[5];
creg c[5];

h q[0]; ...



## Explore other preparation topologies
Switch the `preparation_method` between `star`, `linear`, and `log` to see how fanout depth affects circuit sizes.


In [7]:
for method in ["star", "linear", "log"]:
    demo = GHZBenchmark(
        number_of_qubits=6,
        sample_size=1,
        preparation_method=method,
        measurement_scheme="witness",
        auto_save=False,
    )
    demo.create_benchmark()
    circuits = demo.samples[0]["circuits"]
    lengths = [len(c["qasm"].splitlines()) for c in circuits]
    print(f"{method.capitalize()} topology -> circuit line counts (X/Z): {lengths}")

Star topology -> circuit line counts (X/Z): [25, 19]
Linear topology -> circuit line counts (X/Z): [25, 19]
Log topology -> circuit line counts (X/Z): [25, 19]


In [8]:
print(benchmark.get_all_circuit_ids())
print()
benchmark.get_all_circuits()[0]

['0_witness_x', '0_witness_z', '1_witness_x', '1_witness_z', '2_witness_x', '2_witness_z']



'OPENQASM 2.0;\n\ninclude "qelib1.inc";\n\nqreg q[5];\ncreg c[5];\n\nh q[0];\ncx q[0], q[1];\ncx q[0], q[2];\ncx q[0], q[3];\ncx q[0], q[4];\nh q[0];\nh q[1];\nh q[2];\nh q[3];\nh q[4];\n\nmeasure q[0] -> c[0];\nmeasure q[1] -> c[1];\nmeasure q[2] -> c[2];\nmeasure q[3] -> c[3];\nmeasure q[4] -> c[4];'

## Load benchmark from disk
The repo ships with a saved GHZ benchmark JSON under `notebooks/tutorial_data`. Reload it to simulate sharing a dataset.


In [9]:
benchmark = GHZBenchmark.load_json(r"tutorial_data/ghz_nq5_s3_tutorial.json")

## Attach experimental results
Synthetic witness counts live in `ghz_benchmark_results.json`. Attach them so the benchmark can evaluate fidelities.


In [10]:
import json

with open("tutorial_data/ghz_benchmark_results.json", "r", encoding="utf-8") as fh:
    witness_results = json.load(fh)

In [11]:
benchmark.add_experimental_results(
    witness_results,
    experiment_id="synthetic_ghz_demo",
    platform="simulator",
    experiment_metadata={"backend": "mock-sampler"},
)

[Benchmark] Saved to: tutorial_data\ghz_nq5_s3_20260409T135443_e9d5b419.json


## Inspect attached data


In [12]:
print("experiment_id:", benchmark.experimental_results["experiment_id"])
print("platform:", benchmark.experimental_results["platform"])
list(benchmark.experimental_results["results"].items())[:1]

experiment_id: synthetic_ghz_demo
platform: simulator


[('0_witness_x',
  {'counts': {'00000': 595,
    '00011': 235,
    '01100': 114,
    '10101': 207,
    '11010': 752,
    '11111': 145}})]

## Evaluate the witness benchmark


In [13]:
evaluation = benchmark.evaluate_benchmark()
evaluation

[Benchmark] Saved updated JSON to: tutorial_data\ghz_nq5_s3_20260409T135443_e9d5b419.json


{'method': 'witness',
 'per_sample': {0: {'fidelity_lower_bound': 0.4609375,
   'std_dev': 0.011014774370830644,
   'certified_gt_half': False,
   'confidence': 0.95},
  1: {'fidelity_lower_bound': 0.58984375,
   'std_dev': 0.010868714522160719,
   'certified_gt_half': True,
   'confidence': 0.95},
  2: {'fidelity_lower_bound': 0.39306640625,
   'std_dev': 0.010792910508748713,
   'certified_gt_half': False,
   'confidence': 0.95}},
 'global_metrics': {'mean_fidelity': 0.4812825520833333,
  'min_fidelity': 0.39306640625,
  'max_fidelity': 0.58984375,
  'mean_std_dev': 0.010892133133913358}}

## Access evaluation summaries
Per-sample fidelity bounds and certification flags are stored under `benchmark.experimental_results['evaluation']['per_sample']`.


In [14]:
benchmark.experimental_results["evaluation"]["per_sample"][0]

{'fidelity_lower_bound': 0.4609375,
 'std_dev': 0.011014774370830644,
 'certified_gt_half': False,
 'confidence': 0.95}

## Manual fidelity estimation
Recompute the GHZ fidelity lower bound directly from the witness counts using the helper utilities.


In [15]:
sample0 = benchmark.samples[0]
id_map = {}
for circuit in sample0["circuits"]:
    basis = circuit["metadata"]["basis"]
    id_map[basis] = circuit["circuit_id"]
z_counts = benchmark.experimental_results["results"][id_map["z"]]["counts"]
x_counts = benchmark.experimental_results["results"][id_map["x"]]["counts"]
fidelity, std = evaluate_fidelity(z_counts, x_counts)
print("Fidelity lower bound:", fidelity)
print("Std deviation:", std)
print("Certified > 0.5?:", certify_fidelity_gt_half(fidelity, std))

Fidelity lower bound: 0.4609375
Std deviation: 0.011014774370830644
Certified > 0.5?: False


## Shadow-overlap workflow (optional)
Shadow mode generates randomized two-qubit Pauli probes. Each circuit stores a `basis_map` so you can align shots with the qubits that were measured in X/Y/Z.


In [16]:
shadow_benchmark = GHZBenchmark(
    number_of_qubits=5,
    sample_size=2,
    measurement_scheme="shadow",
    measurement_rounds=20,
    preparation_method="linear",
    shots=1024,
    auto_save=False,
)
shadow_benchmark.create_benchmark()
first_shadow = shadow_benchmark.samples[0]["circuits"][0]
print("circuits generated:", len(shadow_benchmark.get_all_circuit_ids()))
print("basis map for first round:", first_shadow["metadata"]["basis_map"])
print("\n".join(first_shadow["qasm"].splitlines()[:8]), "...")

circuits generated: 40
basis map for first round: {1: 'Y', 0: 'Y'}
OPENQASM 2.0;


qreg q[5];
creg c[5];

h q[0];
cx q[0], q[1]; ...


## Emission preferences (QASM formats and SDK dialects)
Use `QasmEmitterOptions` to switch between QASM2/QASM3 and vendor-specific dialects without changing benchmark logic.


In [33]:
from frontier import QasmEmitterOptions

emitter = QasmEmitterOptions(format="qasm3", target_sdk="braket")
custom = GHZBenchmark(
    number_of_qubits=10,
    sample_size=1,
    measurement_scheme="witness",
    preparation_method="linear",
    emitter_options=emitter,
    auto_save=False,
)
custom.create_benchmark()
print("Emitter options:", custom.emitter_options)
print("QASM header preview:")
print("\n".join(custom.samples[0]["circuits"][0]["qasm"].splitlines()[:8]), "...")

Emitter options: QasmEmitterOptions(format='qasm3', target_sdk='braket', includes=(), float_precision=6, custom_template=None, map={'x': 'x', 'y': 'y', 'z': 'z', 'h': 'h', 's': 's', 'sdg': 'si', 't': 't', 'p': 'p', 'u': 'U', 'cx': 'cnot', 'cy': 'cy', 'cz': 'cz', 'ch': 'ch', 'swap': 'swap', 'rx': 'rx', 'ry': 'ry', 'rz': 'rz', 'measure': 'measure'})
QASM header preview:
OPENQASM 3.0;


qubit[10] q;
bit[10] c;

h q[0];
cnot q[0], q[1]; ...


In [34]:
circ = custom.get_all_circuits()

In [35]:
print(circ[0])

OPENQASM 3.0;


qubit[10] q;
bit[10] c;

h q[0];
cnot q[0], q[1];
cnot q[1], q[2];
cnot q[2], q[3];
cnot q[3], q[4];
cnot q[4], q[5];
cnot q[5], q[6];
cnot q[6], q[7];
cnot q[7], q[8];
cnot q[8], q[9];
h q[0];
h q[1];
h q[2];
h q[3];
h q[4];
h q[5];
h q[6];
h q[7];
h q[8];
h q[9];

c[0] = measure q[0];
c[1] = measure q[1];
c[2] = measure q[2];
c[3] = measure q[3];
c[4] = measure q[4];
c[5] = measure q[5];
c[6] = measure q[6];
c[7] = measure q[7];
c[8] = measure q[8];
c[9] = measure q[9];
